In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (MinMaxScaler,OneHotEncoder,OrdinalEncoder)
from sklearn.decomposition import PCA

import joblib
from pathlib import Path

In [2]:
df = pd.read_csv("../data/eda/campuspulse_akgec_eda.csv")

print(df.shape)

(200000, 37)


In [3]:
df["complaint_date"] = pd.to_datetime(df["complaint_date"],errors="coerce")

In [4]:
df["complaint_date"].dtype

dtype('<M8[ns]')

In [5]:
df["complaint_month"] = df["complaint_date"].dt.month
df["complaint_dayofweek"] = df["complaint_date"].dt.dayofweek
df["complaint_year"] = df["complaint_date"].dt.year

df["is_weekend"] = (df["complaint_dayofweek"] >= 5).astype(int)

In [6]:
df["complaint_time"].head()

0    12:49:29
1    12:21:32
2    16:44:02
3    16:03:21
4    16:38:57
Name: complaint_time, dtype: object

In [7]:
df["complaint_time"] = pd.to_datetime(df["complaint_time"],errors="coerce")

C:\Users\acer\AppData\Local\Temp\ipykernel_15652\2019043508.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["complaint_time"] = pd.to_datetime(df["complaint_time"],errors="coerce")


In [8]:
df["complaint_hour"] = df["complaint_time"].dt.hour

df["time_period"] = pd.cut(df["complaint_hour"],
    bins=[-1, 5, 11, 17, 21, 24],
    labels=["Night","Morning","Afternoon","Evening","Late_Night"])

In [9]:
recent_7_days = df["complaints_last_7_days"].fillna(0)
recent_30_days = df["complaints_last_30_days"].fillna(0)
recurrence = df["recurrence_count"].fillna(0)
affected_users = df["affected_users"].fillna(0)

df["total_recent_complaints"] = recent_7_days + recent_30_days

df["complaint_growth"] = recent_7_days / (recent_30_days + 1)

df["impact_score"] = affected_users * (1 + recurrence)

df["duration_per_user"] = (df["issue_duration_hours"].fillna(0) /(affected_users + 1))

df["repair_cost_per_user"] = (df["estimated_repair_cost"].fillna(0) /(affected_users + 1))

df["equipment_infrastructure_age_gap"] = (df["infrastructure_age_years"].fillna(0) -df["equipment_age_years"].fillna(0))

df["operational_pressure"] = (recent_7_days +recurrence +affected_users / 10)

print("Feature engineering completed.")

Feature engineering completed.


In [10]:
df.drop(columns=["complaint_date", "complaint_time"],inplace=True)

In [11]:
leakage_cols = ["complaint_status",
    "response_time_hours",
    "resolution_time_hours",
    "maintenance_team",
    "resolution_type",
    "actual_expenditure"]

In [12]:
df.drop(columns=leakage_cols, inplace=True)

In [13]:
df.drop(columns=["complaint_id"], inplace=True)

In [14]:
print(df.shape)
print(df.columns.tolist())

(200000, 41)
['complainant_type', 'department', 'year_of_study', 'campus_zone', 'building', 'floor', 'room_lab_no', 'facility_type', 'class_section', 'issue_category', 'issue_subcategory', 'severity', 'affected_users', 'issue_duration_hours', 'previous_similar_complaints', 'complaints_last_7_days', 'complaints_last_30_days', 'recurrence_count', 'equipment_age_years', 'infrastructure_age_years', 'connectivity_impact', 'safety_risk', 'academic_impact', 'operational_impact', 'estimated_repair_cost', 'weather_condition', 'submission_channel', 'priority', 'complaint_month', 'complaint_dayofweek', 'complaint_year', 'is_weekend', 'complaint_hour', 'time_period', 'total_recent_complaints', 'complaint_growth', 'impact_score', 'duration_per_user', 'repair_cost_per_user', 'equipment_infrastructure_age_gap', 'operational_pressure']


In [15]:
X = df.drop(columns="priority")
y = df["priority"]

In [16]:
print("X:", X.shape)
print("y:", y.shape)

X: (200000, 40)
y: (200000,)


In [17]:
print(y.value_counts())

priority
Medium      70197
Low         68808
High        42242
Critical    18753
Name: count, dtype: int64


In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,random_state=42,stratify=y)

In [19]:
print(X_train.shape)
print(X_test.shape)

(160000, 40)
(40000, 40)


In [20]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()

ordinal_cols = ["severity","safety_risk","academic_impact","operational_impact","floor"]

num_cols = [col for col in num_cols if col not in ordinal_cols]

print("Numeric columns:", len(num_cols))
print(num_cols)

Numeric columns: 22
['year_of_study', 'affected_users', 'issue_duration_hours', 'previous_similar_complaints', 'complaints_last_7_days', 'complaints_last_30_days', 'recurrence_count', 'equipment_age_years', 'infrastructure_age_years', 'estimated_repair_cost', 'complaint_month', 'complaint_dayofweek', 'complaint_year', 'is_weekend', 'complaint_hour', 'total_recent_complaints', 'complaint_growth', 'impact_score', 'duration_per_user', 'repair_cost_per_user', 'equipment_infrastructure_age_gap', 'operational_pressure']


In [21]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:", len(cat_cols))
print(cat_cols)

Categorical columns: 18
['complainant_type', 'department', 'campus_zone', 'building', 'floor', 'room_lab_no', 'facility_type', 'class_section', 'issue_category', 'issue_subcategory', 'severity', 'connectivity_impact', 'safety_risk', 'academic_impact', 'operational_impact', 'weather_condition', 'submission_channel', 'time_period']


In [22]:
cat_cols = [col for col in cat_cols if col not in ordinal_cols]

print("Ordinal:", ordinal_cols)
print("Nominal:", cat_cols)

Ordinal: ['severity', 'safety_risk', 'academic_impact', 'operational_impact', 'floor']
Nominal: ['complainant_type', 'department', 'campus_zone', 'building', 'room_lab_no', 'facility_type', 'class_section', 'issue_category', 'issue_subcategory', 'connectivity_impact', 'weather_condition', 'submission_channel', 'time_period']


In [23]:
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", MinMaxScaler())])

In [24]:
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
                     ])

In [25]:
severity_order = ["Low","Medium","High","Critical"]
floor_order = ["Ground","1","2","3","4","5"]
ordinal_categories = [severity_order,severity_order,severity_order,severity_order,floor_order]

ordinal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("encoder", OrdinalEncoder(categories=ordinal_categories,handle_unknown="use_encoded_value",unknown_value=-1)),
                         ("scaler", MinMaxScaler())
                         ])

In [26]:
preprocessor = ColumnTransformer([("num", num_pipe, num_cols),
                                  ("ordinal", ordinal_pipe, ordinal_cols),
                                  ("cat", cat_pipe, cat_cols)
                                  ])

In [27]:
X_train_processed = preprocessor.fit_transform(X_train)

In [28]:
X_test_processed = preprocessor.transform(X_test)

In [29]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(160000, 304)
(40000, 304)


In [30]:
feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))
print(feature_names[:20])

304
['num__year_of_study' 'num__affected_users' 'num__issue_duration_hours'
 'num__previous_similar_complaints' 'num__complaints_last_7_days'
 'num__complaints_last_30_days' 'num__recurrence_count'
 'num__equipment_age_years' 'num__infrastructure_age_years'
 'num__estimated_repair_cost' 'num__complaint_month'
 'num__complaint_dayofweek' 'num__complaint_year' 'num__is_weekend'
 'num__complaint_hour' 'num__total_recent_complaints'
 'num__complaint_growth' 'num__impact_score' 'num__duration_per_user'
 'num__repair_cost_per_user']


In [31]:
X_train_dense = (X_train_processed.toarray() 
                 if hasattr(X_train_processed, "toarray")
                 else X_train_processed)

X_test_dense = (X_test_processed.toarray()
                if hasattr(X_test_processed, "toarray")
                else X_test_processed)

pca = PCA(n_components=0.95,svd_solver="full",random_state=42)

X_train_pca = pca.fit_transform(X_train_dense)
X_test_pca = pca.transform(X_test_dense)

print("Before PCA:", X_train_dense.shape)
print("After PCA:", X_train_pca.shape)

print("Variance retained:",round(pca.explained_variance_ratio_.sum(), 4))

print("Number of PCA components:",pca.n_components_)

Before PCA: (160000, 304)
After PCA: (160000, 152)
Variance retained: 0.9506
Number of PCA components: 152


In [32]:
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

joblib.dump(preprocessor, "../ml/models/classification/campuspulse_preprocessor.pkl")

joblib.dump(pca, "../ml/models/classification/campuspulse_pca.pkl")

print("Preprocessor and PCA saved successfully.")

Preprocessor and PCA saved successfully.


## Preprocessing Summary

- Removed the complaint identifier.
- Removed columns that could cause target leakage.
- Created useful date and time features.
- Created additional complaint, impact, duration, cost, and operational features.
- Separated `priority` as the target variable.
- Used an 80/20 stratified train-test split.
- Numerical missing values are filled using the median.
- Categorical missing values are filled using the most frequent value.
- Nominal categorical features are encoded with `OneHotEncoder`.
- Ordinal features are encoded according to their defined order.
- Numerical and encoded features are scaled with `MinMaxScaler`.
- `ColumnTransformer` keeps the preprocessing steps organized in one pipeline.
- PCA is applied after preprocessing while retaining 95% of the variance.
- The fitted preprocessor and PCA model are saved for later prediction.

In [33]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "campuspulse_akgec_processed.csv"

df.to_csv(output_path, index=False)

print(f"Processed dataset exported to: {output_path}")
print(f"Shape: {df.shape}")

Processed dataset exported to: ..\data\processed\campuspulse_akgec_processed.csv
Shape: (200000, 41)
